In [1]:
import PF_wrapper as PF
import makemissing
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import MDS
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import KNNImputer

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#from sklearn.datasets import make_blobs

import geomstats.backend as gs
import geomstats.visualization as visualization
from geomstats.geometry.hypersphere import Hypersphere
from geomstats.geometry.special_orthogonal import SpecialOrthogonal

# Code for Experiment 4 functions

In [3]:
#& Data Constants
SEEDS = [61, 737, 821, 161, 346, 78, 2, 67, 102, 982]

In [4]:
#& Generate Data Functions
def create_3d_sphere_data(seed):
    """Creates two clusters on a 3d sphere"""

    np.random.seed(seed)
    gs.random.seed(seed)

    sphere = Hypersphere(dim=2)
    cluster = sphere.random_von_mises_fisher(kappa=20, n_samples=150)

    SO3 = SpecialOrthogonal(3, equip=False)
    rotation1 = SO3.random_uniform()
    rotation2 = SO3.random_uniform()

    cluster_1 = cluster @ rotation1
    cluster_2 = cluster @ rotation2

    #Create labels
    labels_1 = np.zeros(cluster_1.shape[0])
    labels_2 = np.ones(cluster_2.shape[0])
    labels = np.concatenate([labels_1, labels_2])

    #Combine data
    data = np.concatenate([cluster_1, cluster_2], axis=0)
    
    return data, labels, sphere

def to_spherical_coords(data):
    """Convert 3D points on the unit sphere to 2D spherical coordinates (polar, azimuthal)."""
    x, y, z = data[:, 0], data[:, 1], data[:, 2]
    r = np.linalg.norm(data, axis=1)
    r = np.where(r == 0, 1e-8, r)
    polar = np.arccos(z / r)
    azimuth = np.arctan2(y, x)
    return np.vstack((polar, azimuth)).T

In [25]:
X, y, man = create_3d_sphere_data(SEEDS[0])

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [27]:
KNN = KNeighborsClassifier()

In [28]:
KNN.fit(X_train, y_train)
knnScore = KNN.score(X_test, y_test)
print(knnScore)

1.0


In [29]:
trainDS = pd.DataFrame(data=X_train)
trainDS.insert(loc=0, column='labels', value=y_train.astype(int))

testDS = pd.DataFrame(data=X_test)
testDS.insert(loc=0, column='labels', value=y_test.astype(int))

In [30]:
trainDS.to_csv("sphere_train.tsv",sep=",",header=False,index=False)
testDS.to_csv("sphere_test.tsv",sep=",",header=False,index=False)

In [31]:
PF.train("sphere_train.tsv", "sphere_test.tsv", model_name="Spartacus", 
         distances=['euclidean', 'manhattan'], exists_testlabels=True, num_trees=100)


0:3mb
finished in 0:0:0.011

0:4mb
finished in 0:0:0.002

-----------------Repetition No: 1 (sphere_train.tsv)   -----------------
Using: 2 MB, Free: 18 MB, Allocated Pool: 20 MB, Max Available: 1024 MB
core.ProximityForestResult@4edde6e5
0.1.2.3.4.5.6.7.8.9.10.11.12.13.14.15.16.17.18.19.20.21.22.23.24.25.26.27.28.29.30.31.32.33.34.35.36.37.38.39.40.41.42.43.44.45.46.47.48.49.50.51.52.53.54.55.56.57.58.59.60.61.62.63.64.65.66.67.68.69.70.71.72.73.74.75.76.77.78.79.80.81.82.83.84.85.86.87.88.89.90.91.92.93.94.95.96.97.98.99.
Using: 13 MB, Free: 120 MB, Allocated Pool: 133 MB, Max Available: 1024 MB
**
Training Time: 122.613332ms (0:0:0.122)
Prediction Time: 7.727473ms (0:0:0.007)
Correct(TP+TN): 148 vs Incorrect(FP+FN): 2
Score: 0.9866666666666667
Error Rate: 0.013333333333333308
REPEAT:1 ,sphere_train.tsv, 0.9866666666666667, 122.613332, 7.727473, 4.53


In [33]:
Xm = makemissing.remove_mcar(pd.DataFrame(X), pct=0.5, random_state=0)

In [34]:
Xm_train, Xm_test, y_train, y_test = train_test_split(Xm, y, test_size=0.5, random_state=42)

In [35]:
Xm_train.to_csv("sphere_train_m.tsv",sep=",",header=False,index=False)
Xm_test.to_csv("sphere_test_m.tsv",sep=",",header=False,index=False)

In [36]:
pd.DataFrame(y_train).to_csv("sphere_train_labels.csv",sep=",",header=False,index=False)
pd.DataFrame(y_test).to_csv("sphere_test_labels.csv",sep=",",header=False,index=False)

In [44]:
PF.train("sphere_train_m.tsv", "sphere_test_m.tsv", model_name="Spartacus_m", num_trees=100,
         distances=['euclidean'], train_labels="sphere_train_labels.csv", test_labels="sphere_test_labels.csv",
         impute_training_data=True, impute_testing_data=True, return_imputed_training=True, return_imputed_testing=True,
        output_directory="sphere_output", impute_iterations=5, knn_distances=['euclidean'], initial_imputer="knn")


0:3mb
finished in 0:0:0.011

0:5mb
finished in 0:0:0.002

-----------------Repetition No: 1 (sphere_train_m.tsv)   -----------------
Using: 2 MB, Free: 22 MB, Allocated Pool: 24 MB, Max Available: 1024 MB
Imputing the training set...
core.ProximityForestResult@246b179d
0.1.2.3.4.5.6.7.8.9.10.11.12.13.14.15.16.17.18.19.20.21.22.23.24.25.26.27.28.29.30.31.32.33.34.35.36.37.38.39.40.41.42.43.44.45.46.47.48.49.50.51.52.53.54.55.56.57.58.59.60.61.62.63.64.65.66.67.68.69.70.71.72.73.74.75.76.77.78.79.80.81.82.83.84.85.86.87.88.89.90.91.92.93.94.95.96.97.98.99.
Using: 56 MB, Free: 79 MB, Allocated Pool: 135 MB, Max Available: 1024 MB
core.ProximityForestResult@2d8e6db6
0.1.2.3.4.5.6.7.8.9.10.11.12.13.14.15.16.17.18.19.20.21.22.23.24.25.26.27.28.29.30.31.32.33.34.35.36.37.38.39.40.41.42.43.44.45.46.47.48.49.50.51.52.53.54.55.56.57.58.59.60.61.62.63.64.65.66.67.68.69.70.71.72.73.74.75.76.77.78.79.80.81.82.83.84.85.86.87.88.89.90.91.92.93.94.95.96.97.98.99.
Using: 48 MB, Free: 87 MB, Allocated 

In [45]:
PF.predict("sphere_output/Spartacus_m", "sphere_output/sphere_test_m.tsv", test_labels="sphere_test_labels.csv",
           output_directory="Sphere_test_output", return_predictions=True)


0:21mb
finished in 0:0:0.010
**
Training Time: 57.058793ms (0:0:0.057)
Prediction Time: 25.199308ms (0:0:0.025)
Correct(TP+TN): 132 vs Incorrect(FP+FN): 18
Score: 0.88
Error Rate: 0.12
REPEAT:1 ,sphere_train_m.tsv, 0.88, 57.058793, 25.199308, 8.72


In [39]:
imputer = KNNImputer()

In [40]:
imputer.fit(Xm_train)

,missing_values,nan
,n_neighbors,5
,weights,'uniform'
,metric,'nan_euclidean'
,copy,True
,add_indicator,False
,keep_empty_features,False


In [41]:
imputed_train = imputer.transform(Xm_train)
imputed_test = imputer.transform(Xm_test)

In [42]:
KNN2 = KNeighborsClassifier()
KNN2.fit(imputed_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [43]:
KNN2.score(imputed_test, y_test)

0.8466666666666667